# Capítulo 6 · RAG avanzado

Mejorar la recuperación de un sistema RAG y medir cuánto aporta cada pieza: BM25, búsqueda híbrida con fusión por rango recíproco, reordenación con un *cross-encoder*, y reescritura de la consulta y HyDE con el modelo de lenguaje. La base de conocimiento tiene 82 fragmentos (preguntas frecuentes, condiciones y catálogo) y la prueba, 40 consultas de cuatro tipos.

**Antes de empezar, activa la GPU:** menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*. El cuaderno carga tres modelos: el de lenguaje, el de embeddings y el de reordenación (unos 2,3 GB). Sin las variantes que usan el modelo de lenguaje, tarda alrededor de un minuto.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

📘 El libro, en Amazon: [tapa blanda](https://www.amazon.es/dp/B0HMXCBCTN) · [Kindle](https://www.amazon.es/dp/B0HMXG3NXM)

## 1. Instalar BM25, cargar los modelos y los datos

In [ ]:
!pip install -q rank_bm25

In [ ]:
"""Capítulo 6: RAG avanzado. Búsqueda híbrida, reordenación y reescritura."""
import json
import re
import unicodedata
import urllib.request
from pathlib import Path

import numpy as np
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_GPU = "Qwen/Qwen2.5-1.5B-Instruct"  # con la GPU T4 gratuita de Colab
MODELO_CPU = "Qwen/Qwen3-0.6B"             # sin GPU: más pequeño y más rápido
MODELO = None                              # None: elegir según el entorno
MODELO_EMBEDDINGS = "intfloat/multilingual-e5-base"
MODELO_REORDENACION = "BAAI/bge-reranker-v2-m3"
DATOS = Path("datos")
REPOSITORIO = "https://raw.githubusercontent.com/HernanDiaz/llm-engineering/main/"



def cargar(nombre=None, con_lenguaje=True):
    """Los modelos de lenguaje, de embeddings y de reordenación."""
    dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
    if nombre is None:
        nombre = MODELO_GPU if dispositivo == "cuda" else MODELO_CPU
    tok = modelo = None
    if con_lenguaje:
        tok = AutoTokenizer.from_pretrained(nombre)
        modelo = AutoModelForCausalLM.from_pretrained(nombre, dtype=torch.float32)
        modelo.to(dispositivo).eval()
    embeddings = SentenceTransformer(MODELO_EMBEDDINGS, device=dispositivo)
    reordenador = CrossEncoder(MODELO_REORDENACION, device=dispositivo)
    print(f"Modelos en {dispositivo}")
    return tok, modelo, embeddings, reordenador


def descargar(archivo):
    """Un archivo de la carpeta datos; en Colab, se descarga del repositorio."""
    ruta = DATOS / archivo
    if not ruta.exists():
        DATOS.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORIO + ruta.as_posix(), ruta)
    return ruta.read_text(encoding="utf-8")


def base_de_conocimiento():
    """Preguntas frecuentes, condiciones generales y catálogo, con un
    identificador por fragmento."""
    fragmentos = []
    for p in json.loads(descargar("preguntas_frecuentes.json"))["preguntas"]:
        fragmentos.append({"id": f"faq-{p['id']}",
                           "texto": p["pregunta"] + " " + p["respuesta"]})
    condiciones = descargar("condiciones_generales.md")
    for seccion in re.split(r"\n(?=## )", condiciones)[1:]:
        titulo, texto = seccion.strip().split("\n", 1)
        numero = titulo.split(".")[0][3:]
        fragmentos.append({"id": f"cond-{numero}",
                           "texto": titulo[3:] + ". " + texto.strip()})
    for p in json.loads(descargar("catalogo.json")):
        fragmentos.append({"id": p["ref"], "texto": (
            f"{p['nombre']} (referencia {p['ref']}), {p['categoria']}, "
            f"{p['precio']} euros. {p['descripcion']}")})
    return fragmentos

In [ ]:
tok, modelo, embeddings, reordenador = cargar(MODELO)
fragmentos = base_de_conocimiento()
textos = {f["id"]: f["texto"] for f in fragmentos}
consultas = json.loads(descargar("consultas_recuperacion.json"))
print(len(fragmentos), "fragmentos y", len(consultas), "consultas")
print(textos["UR-5310"])

## 2. Dos buscadores

In [ ]:
class BuscadorSemantico:
    """Embeddings, como en el capítulo 4. Devuelve los identificadores ordenados."""

    def __init__(self, fragmentos):
        self.ids = [f["id"] for f in fragmentos]
        self.matriz = embeddings.encode(["passage: " + f["texto"] for f in fragmentos],
                                        normalize_embeddings=True)

    def ordenar(self, consulta, prefijo="query: "):
        vector = embeddings.encode([prefijo + consulta], normalize_embeddings=True)
        return [self.ids[i] for i in np.argsort(-(self.matriz @ vector[0]))]


PALABRAS_VACIAS = set(
    "a al algo con de del el en es la las lo los me mi mis no o para por que se "
    "si su sus te tu un una unos y ya os le les como cuando donde cual cuanto "
    "cuanta cuantos cuantas hay tiene tienen tengo puedo".split())


def palabras(texto):
    """Minúsculas, sin tildes y sin palabras vacías. Las referencias como
    UR-5310 se conservan enteras."""
    texto = "".join(c for c in unicodedata.normalize("NFD", texto.lower())
                    if unicodedata.category(c) != "Mn")
    return [p for p in re.findall(r"[a-z]{2}-\d{4}|[a-z0-9]+", texto)
            if p not in PALABRAS_VACIAS]


class BuscadorBM25:
    """Búsqueda por palabras con BM25."""

    def __init__(self, fragmentos):
        self.ids = [f["id"] for f in fragmentos]
        self.bm25 = BM25Okapi([palabras(f["texto"]) for f in fragmentos])

    def ordenar(self, consulta, con_ceros=False):
        """Los fragmentos sin ninguna palabra en común con la consulta tienen
        puntuación 0, y su orden no significa nada: se quitan."""
        puntos = self.bm25.get_scores(palabras(consulta))
        orden = np.argsort(-puntos, kind="stable")
        return [self.ids[i] for i in orden if con_ceros or puntos[i] > 0]

In [ ]:
semantico, bm25 = BuscadorSemantico(fragmentos), BuscadorBM25(fragmentos)
consulta = "¿Qué precio tiene la referencia UR-5310?"
print("Palabras:", palabras(consulta))
print("Embeddings:", semantico.ordenar(consulta)[:3])
print("BM25:", bm25.ordenar(consulta)[:3])

## 3. Fusionar y reordenar

In [ ]:
def fusionar(*rankings, k=60):
    """Fusión por rango recíproco (RRF): cada lista suma 1 / (k + posición)."""
    puntos = {}
    for ranking in rankings:
        for posicion, ident in enumerate(ranking, 1):
            puntos[ident] = puntos.get(ident, 0) + 1 / (k + posicion)
    return sorted(puntos, key=puntos.get, reverse=True)


def reordenar(consulta, candidatos, textos, n=20):
    """Un modelo de reordenación puntúa cada par (consulta, fragmento) de los
    n primeros candidatos; el resto se queda detrás, en su orden."""
    primeros = candidatos[:n]
    puntos = reordenador.predict([(consulta, textos[i]) for i in primeros])
    return [primeros[i] for i in np.argsort(-puntos)] + candidatos[n:]

In [ ]:
consulta = "Si no estoy en casa cuando viene el mensajero, ¿qué pasa?"
hibrida = fusionar(semantico.ordenar(consulta), bm25.ordenar(consulta))
print("Híbrida:", hibrida[:3])
print("Reordenada:", reordenar(consulta, hibrida, textos)[:3])

## 4. Reescribir la consulta

In [ ]:
def preguntar(sistema, usuario, ejemplos=(), max_tokens=120):
    """Envía una conversación al modelo y devuelve su respuesta como texto."""
    mensajes = [{"role": "system", "content": sistema}]
    for pregunta, respuesta in ejemplos:
        mensajes += [{"role": "user", "content": pregunta},
                     {"role": "assistant", "content": respuesta}]
    mensajes.append({"role": "user", "content": usuario})
    texto = tok.apply_chat_template(mensajes, tokenize=False,
                                    add_generation_prompt=True,
                                    enable_thinking=False)
    entrada = tok(texto, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=max_tokens,
                                 do_sample=False, temperature=None, top_p=None,
                                 top_k=None, pad_token_id=tok.eos_token_id)
    nuevos = salida[0, entrada["input_ids"].shape[1]:]
    respuesta = tok.decode(nuevos, skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", respuesta, flags=re.S).strip()


REESCRIBIR = ("Eres el buscador de Ciclos Norte, una tienda de bicicletas. "
              "Reescribe la consulta del cliente como una frase de búsqueda clara "
              "y completa, con las palabras que usaría la tienda. Conserva las "
              "referencias y los nombres de producto. No respondas a la consulta: "
              "escribe solo la frase de búsqueda.")


HIPOTETICA = ("Eres el redactor de la web de Ciclos Norte, una tienda de "
              "bicicletas. Escribe en dos frases el texto de la web o de las "
              "condiciones de venta que respondería a la consulta del cliente. "
              "Si no conoces los datos exactos, invéntalos.")


EJEMPLOS_REESCRIBIR = [
    ("se me ha pinchado la rueda nueva",
     "¿Cubre la garantía un pinchazo en una cubierta nueva?"),
    ("lo del envío a las islas",
     "¿Cuánto cuesta y cuánto tarda el envío a Baleares y Canarias?"),
    ("¿La CA-4100 tiene visera?",
     "¿Tiene visera el casco de referencia CA-4100?")]


EJEMPLOS_HIPOTETICA = [
    ("se me ha pinchado la rueda nueva",
     "La garantía no cubre el desgaste normal de piezas como cubiertas y "
     "cámaras, ni los pinchazos. Sí cubre los defectos de fabricación durante "
     "2 años."),
    ("¿La CA-4100 tiene visera?",
     "Casco Trail (referencia CA-4100), cascos, 89 euros. Casco de montaña con "
     "visera ajustable y protección en la nuca.")]


def reescribir(consulta):
    """Una consulta más clara y completa, escrita por el modelo."""
    return preguntar(REESCRIBIR, consulta, EJEMPLOS_REESCRIBIR)


def hipotetica(consulta):
    """HyDE: una respuesta inventada, que se parece más a los documentos que
    la propia consulta."""
    return preguntar(HIPOTETICA, consulta, EJEMPLOS_HIPOTETICA)

In [ ]:
for consulta in ["lo de pagar poco a poco", "la caja vino hecha polvo"]:
    print(consulta)
    print("   Reescrita:", reescribir(consulta))
    print("   HyDE:", hipotetica(consulta))

## 5. Medir

In [ ]:
def medir(rankings, consultas, ks=(1, 3, 5)):
    """Acierto en los k primeros (recall@k) y rango recíproco medio (MRR)."""
    resultado = {f"@{k}": 0 for k in ks}
    rr = 0
    for ranking, c in zip(rankings, consultas):
        posiciones = [i for i, ident in enumerate(ranking, 1)
                      if ident in c["correctos"]]
        primera = min(posiciones) if posiciones else None
        for k in ks:
            resultado[f"@{k}"] += primera is not None and primera <= k
        rr += 1 / primera if primera else 0
    resultado["MRR"] = round(rr / len(consultas), 3)
    return resultado


def por_tipo(rankings, consultas, k=3):
    """Aciertos en los k primeros, separados por tipo de consulta."""
    tipos = {}
    for ranking, c in zip(rankings, consultas):
        acierto = any(i in c["correctos"] for i in ranking[:k])
        total = tipos.setdefault(c["tipo"], [0, 0])
        total[0] += acierto
        total[1] += 1
    return {t: f"{a}/{n}" for t, (a, n) in tipos.items()}

Todas las variantes. Para no usar el modelo de lenguaje, llama a `ejecutar(con_reescritura=False)`.

In [ ]:
def ejecutar(con_reescritura=True):
    fragmentos = base_de_conocimiento()
    textos = {f["id"]: f["texto"] for f in fragmentos}
    consultas = json.loads(descargar("consultas_recuperacion.json"))
    semantico, bm25 = BuscadorSemantico(fragmentos), BuscadorBM25(fragmentos)
    print(len(fragmentos), "fragmentos,", len(consultas), "consultas")
    sem = [semantico.ordenar(c["consulta"]) for c in consultas]
    pal = [bm25.ordenar(c["consulta"]) for c in consultas]
    ceros = [bm25.ordenar(c["consulta"], con_ceros=True) for c in consultas]
    hib = [fusionar(s, p) for s, p in zip(sem, pal)]
    reo = [reordenar(c["consulta"], h, textos) for c, h in zip(consultas, hib)]
    metodos = {"Embeddings": sem, "BM25": pal,
               "Híbrida con los ceros de BM25": [fusionar(s, p)
                                                 for s, p in zip(sem, ceros)],
               "Híbrida": hib, "Híbrida + reordenación": reo}
    if con_reescritura:
        nuevas = [reescribir(c["consulta"]) for c in consultas]
        metodos["Reescritura + híbrida + reord."] = [
            reordenar(n, fusionar(semantico.ordenar(n), bm25.ordenar(n)), textos)
            for n in nuevas]
        hyde = [hipotetica(c["consulta"]) for c in consultas]
        solo_hyde = [semantico.ordenar(h, "passage: ") for h in hyde]
        metodos["Solo HyDE"] = solo_hyde
        metodos["Híbrida + HyDE + reord."] = [
            reordenar(c["consulta"], fusionar(s, p, h), textos)
            for c, s, p, h in zip(consultas, sem, pal, solo_hyde)]
    resultados = {}
    for nombre, rankings in metodos.items():
        resultados[nombre] = {**medir(rankings, consultas),
                              "por_tipo": por_tipo(rankings, consultas),
                              "primeros": [r[:5] for r in rankings]}
        print(f"{nombre:32s}", {k: v for k, v in resultados[nombre].items()
                                 if k != "primeros"})
    if con_reescritura:
        resultados["consultas_reescritas"] = nuevas
        resultados["respuestas_hipoteticas"] = hyde
    return resultados

In [ ]:
resultados = ejecutar()

Los fallos del mejor método, con los tres primeros fragmentos que devuelve:

In [ ]:
mejor = resultados["Híbrida + reordenación"]["primeros"]
for c, primeros in zip(consultas, mejor):
    if not any(i in c["correctos"] for i in primeros[:3]):
        print(c["consulta"], "→", primeros[:3], "| correctos:", c["correctos"])

## Ejercicios

1. Explica con tus palabras por qué los embeddings fallan con las referencias de producto y por qué BM25 falla con las consultas coloquiales.

2. Calcula a mano la puntuación RRF, con k = 60, de un fragmento que sale tercero en los embeddings y quinto en BM25, y de otro que sale primero en los embeddings y no aparece en BM25. ¿Cuál queda por delante?

3. Prueba la fusión con k = 1 y con k = 1000. ¿Qué pasa con los resultados? ¿Por qué?

4. Cambia el número de candidatos que se reordenan, `n`, a 5, 10 y 40. ¿Cómo cambian los aciertos? ¿Y el tiempo?

5. Reordena directamente los resultados de los embeddings, sin búsqueda híbrida, y comprueba que obtienes lo mismo que la búsqueda híbrida con reordenación. Repite la comparación reordenando solo 5 candidatos. ¿Cuándo aporta BM25?

6. Mejora las descripciones de los candados y de la bicicleta de enduro para que digan para qué sirven, con las palabras de un cliente, y repite la medida. ¿Se encuentran ahora?

7. Añade a cada fragmento del catálogo su categoría y su precio como metadatos. Escribe una función que, para consultas como «bicicleta eléctrica de menos de 2000 euros», filtre primero por categoría y precio y después busque entre los fragmentos que quedan. ¿Cómo extraerías las condiciones de la consulta? (Pista: capítulo 3).

8. Escribe cinco consultas que dependan de una conversación anterior, como «¿y en talla 54?» después de preguntar por la Aero 3. ¿Las encuentra algún método? Reescríbelas tú a mano y compara.

9. Conecta la mejor recuperación con el asistente del capítulo 5 y repite su evaluación con las veinte preguntas. ¿Mejoran las respuestas? ¿Se recupera ahora el fragmento de la pregunta sobre los gastos de envío de una devolución parcial?

10. **Informe de recuperación.** Escribe un informe breve, de unas diez líneas, que recomiende una configuración de recuperación para Ciclos Norte, justificada con las cifras de tu ejecución y teniendo en cuenta el tiempo de cada variante.

## Referencias

Modelos [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), [multilingual-e5-base](https://huggingface.co/intfloat/multilingual-e5-base) y [bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3). BM25: [Robertson y Zaragoza (2009)](https://doi.org/10.1561/1500000019). RRF: Cormack, Clarke y Büttcher (2009). HyDE: [Gao y colaboradores (2022)](https://arxiv.org/abs/2212.10496). Lección 11.07 y proyectos 19.64 a 19.69 de [AI Engineering from Scratch](https://github.com/rohitg00/ai-engineering-from-scratch). La tienda Ciclos Norte, su catálogo y sus documentos son ficticios. Código y experimentos propios.